# LeetCode 67 — Add Binary | Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** String · hệ nhị phân · carry  
**Bài gốc:** https://leetcode.com/problems/add-binary/

Notebook này dành cho người **mới bắt đầu**. Đọc các ô từ trên xuống: trước hết hiểu **vì sao**, sau đó chạy code nhỏ để tự thấy **cách làm**.

**Khung hàm (signature) cần nộp LeetCode:**

```python
class Solution:
    def addBinary(self, a: str, b: str) -> str:
        ...
```

**Sau bài học, bạn có thể:**
- Đọc được số hệ 2 và bảng cộng 0/1.
- Tự cộng hai chuỗi nhị phân từ phải sang trái với carry.
- Giải thích vì sao thời gian và bộ nhớ O(max(n,m)); test cả số nhớ cuối.

**Cách sử dụng:** chạy từng code cell theo thứ tự bằng `Shift + Enter`. Những ô có `TODO` là bài tự luyện: chỉ định nghĩa hàm hoặc chứa lời nhắc kiểm thử, **không được gọi tự động** khi chưa làm xong.

> Lộ trình: hiểu đề → Python căn bản → cách trực giác → phát hiện nút thắt → cải tiến → trace → nộp LeetCode → test → lỗi → vận dụng → bài tập → đáp án → checklist.

## 1. Bài toán: cộng hai số NHỊ PHÂN

Cho hai chuỗi `a` và `b`, mỗi ký tự là `'0'` hoặc `'1'`. Hãy trả **chuỗi biểu diễn tổng trong hệ nhị phân**.

| a | b | đáp án | vì sao |
|---|---|---|---|
| `"11"` | `"1"` | `"100"` | 3 + 1 = 4 |
| `"1010"` | `"1011"` | `"10101"` | 10 + 11 = 21 |
| `"0"` | `"0"` | `"0"` | 0 + 0 = 0 |

Đề yêu cầu **string**, không phải số nguyên hay list. Không được trả `'0b100'` — đó là tiền tố hiển thị của Python, **không phải** format LeetCode yêu cầu.

In [ ]:
print("11 (nhị phân) =",int("11",2),"(thập phân)")
print("1010 (nhị phân) =",int("1010",2))
print("4 (thập phân) ở nhị phân =",bin(4))

## 2. Hệ cơ số 2 hoạt động thế nào?

Hệ thập phân có 10 chữ số `0..9`, mỗi hàng nhân 10. Hệ nhị phân (binary) chỉ có `0`, `1`, các hàng từ phải sang trái là 1, 2, 4, 8, 16, ...

Ví dụ `1011₂ = 1×8 + 0×4 + 1×2 + 1×1 = 11₁₀`. `int(s,2)` là công cụ Python đọc chuỗi nhị phân; ta dùng để **kiểm chứng**, không cần trong lời giải chính.

In [ ]:
s="1011"
weights=[8,4,2,1]
for bit,weight in zip(s,weights):
    print(bit,"x",weight,"=",int(bit)*weight)
print("Tổng:",int(s,2))

## 3. Bảng cộng hai bit và số nhớ

Một **bit** là chữ số 0 hoặc 1. Khi cộng hai bit và `carry` (số nhớ), tổng có thể là 0,1,2,3.

| a_bit | b_bit | carry | total | bit kết quả | carry mới |
|---:|---:|---:|---:|---:|---:|
| 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 | 1 | 0 |
| 1 | 1 | 0 | 2 | 0 | 1 |
| 1 | 1 | 1 | 3 | 1 | 1 |

**Quy tắc:** `result_bit = total % 2`, `carry = total // 2`. Dấu `%` là phần dư; `//` là chia lấy phần nguyên.

In [ ]:
for total in [0,1,2,3]:
    print(total,"-> bit",total%2,"; nhớ",total//2)

## 4. Python cơ sở: ký tự '1' khác số 1

`'1'` là **string**, không phải số nguyên. `int('1')` chuyển ký tự thành 1. `str(1)` làm ngược lại. Ta cần chuyển từng ký tự thành số trước khi cộng, và sau cùng chuyển bit kết quả về string để ghép.

In [ ]:
ch="1"
print("kiểu gốc:",type(ch).__name__)
print("đổi thành int:",int(ch),type(int(ch)).__name__)
print("đổi 0 thành chuỗi:",repr(str(0)))

## 5. Trực giác: dùng Python cộng số nguyên

Với các chuỗi ngắn, `int(a,2) + int(b,2)` tạo tổng thập phân và `bin(sum)[2:]` trả chuỗi nhị phân. `[2:]` bỏ hai ký tự đầu `0b`.

Cách này **hợp lệ trong Python**, nhưng nhiều ngôn ngữ không chứa nổi số nhị phân rất dài trong một biến số nguyên; mục tiêu bài học là mô phỏng phép cộng bằng tay trên từng bit.

In [ ]:
a,b="11","1"
total=int(a,2)+int(b,2)
print("tổng thập phân:",total)
print("bin(total):",bin(total))
print("bỏ '0b':",bin(total)[2:])

## 6. Insight: bắt đầu từ bit cuối (bên phải)

Cũng như cộng thập phân, ta cộng **hàng đơn vị trước**, rồi mang carry sang hàng bên trái. Hai chuỗi có thể có độ dài **khác nhau**, nên cần hai chỉ số độc lập:

- `i = len(a)-1` trỏ vào bit cuối của a.
- `j = len(b)-1` trỏ vào bit cuối của b.
- Mỗi lần lấy bit nếu chỉ số còn >=0; nếu đã vượt trái thì coi bit của chuỗi đó là 0.

Còn phải xử lý khi **một chuỗi đã hết** nhưng chuỗi kia vẫn có bit, hoặc khi cả hai hết mà carry vẫn là 1.

In [ ]:
a,b="1010","11"
i,j=len(a)-1,len(b)-1
print("a cuối:",a[i],"b cuối:",b[j])
print("a trước đó:",a[i-1],"b trước đó:",b[j-1])

## 7. `while` với ba điều kiện và `or`

Vòng lặp phải tiếp tục chừng nào **còn bit ở a**, **còn bit ở b**, **hoặc còn carry**:

`while i >= 0 or j >= 0 or carry:`

Trong Python, số nguyên 0 được hiểu là False; 1 được hiểu là True. `or` nghĩa là chỉ cần **một vế đúng**. Điều này đảm bảo không quên bit nhớ cuối: `1 + 1 = 10`, không được chỉ trả `0`.

In [ ]:
i,j,carry=-1,-1,1
print("Đã hết cả hai chuỗi nhưng còn carry:",i>=0 or j>=0 or carry)
carry=0
print("Đã hết và không còn carry:",bool(i>=0 or j>=0 or carry))

## 8. Xây kết quả ngược rồi đảo lại

Ta xử lý từ phải sang trái, nên những bit tìm được là **từ cuối về đầu**. `result.append(bit)` nối từng bit vào **cuối list**, hiệu quả hơn cộng chuỗi trong nhiều lượt.

Cuối cùng: `''.join(reversed(result))` ghép các ký tự theo thứ tự đảo ngược. `join` nối các string trong list; `reversed` đọc từ cuối list về đầu.

In [ ]:
result=["1","0","1"]   # thứ tự tìm được từ phải sang trái
print("list tạm:",result)
print("chuỗi đúng thứ tự:","".join(reversed(result)))

## 9. Code chính: cộng từng bit kèm carry

Ở mỗi lượt:

1. Khởi tạo `total=carry`.
2. Nếu i còn hợp lệ, cộng bit `int(a[i])` và giảm i; tương tự với j.
3. Lấy `total % 2` làm bit kết quả, `total // 2` làm carry.
4. Thêm bit vào list; khi hết mới đảo lại.

Cách này không cần tạo số nguyên lớn.

In [ ]:
def add_binary_manual(a:str,b:str)->str:
    i,j=len(a)-1,len(b)-1
    carry=0
    result=[]
    while i>=0 or j>=0 or carry:
        total=carry
        if i>=0:
            total+=int(a[i])
            i-=1
        if j>=0:
            total+=int(b[j])
            j-=1
        result.append(str(total%2))
        carry=total//2
    return "".join(reversed(result))

print(add_binary_manual("11","1"))
print(add_binary_manual("1010","1011"))

## 10. Bảng trace bằng tay: `1010` + `1011`

| Lượt (từ phải) | a bit | b bit | carry trước | total | bit lưu | carry sau |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 0 | 1 | 0 | 1 | 1 | 0 |
| 2 | 1 | 1 | 0 | 2 | 0 | 1 |
| 3 | 0 | 0 | 1 | 1 | 1 | 0 |
| 4 | 1 | 1 | 0 | 2 | 0 | 1 |
| 5 | hết | hết | 1 | 1 | 1 | 0 |

List tạo được theo thứ tự là `[1,0,1,0,1]`; đảo lại vẫn là `10101` (trường hợp này đối xứng). Với ví dụ `11 + 1`, list tạm là `[0,0,1]` → đảo ra `100`, sẽ thấy rõ lý do phải đảo.

In [ ]:
def trace_binary(a,b):
    i,j=len(a)-1,len(b)-1;carry=0;bits=[];step=0
    print("a=",a,"b=",b)
    while i>=0 or j>=0 or carry:
        step+=1;ca=carry
        x=int(a[i]) if i>=0 else 0
        y=int(b[j]) if j>=0 else 0
        total=x+y+carry
        bits.append(str(total%2));carry=total//2
        print(f"Bước {step}: a_bit={x}, b_bit={y}, carry_cũ={ca}, total={total}, bit={bits[-1]}, carry_mới={carry}")
        i-=1;j-=1
    ans="".join(reversed(bits));print("Kết quả:",ans)
    return ans
assert trace_binary("1010","1011")=="10101"

## 11. Vì sao thuật toán đúng?

**Bất biến:** sau mỗi lượt, các bit đã ghi trong `result` là **phần đuôi** đúng của tổng (ghi theo thứ tự ngược); `carry` là số nhớ cần cộng vào cột kế tiếp. Mỗi lượt áp dụng đúng phép chia lấy dư và lấy nguyên cho tổng 0..3, vì thế bit mới đúng. Vòng lặp chỉ dừng khi hết bit ở cả hai đầu vào **và** không còn carry, nên không mất dữ liệu.

## 12. Độ phức tạp

Gọi `n=len(a)`, `m=len(b)`.

- **Time:** O(max(n,m)) vì mỗi lượt xử lý tối đa 1 bit của từng chuỗi, nhiều nhất max(n,m)+1 lượt. Ghép/đảo chuỗi cũng tuyến tính.
- **Space:** O(max(n,m)) vì cần lưu list `result` và trả chuỗi đầu ra. Bỏ bộ nhớ cho output thì phương án này vẫn dùng list phụ O(max(n,m)).

Mỗi phép đổi `int('0')` hoặc `int('1')` và thao tác cộng là hằng số.

In [ ]:
a="1"*200
b="1"
ans=add_binary_manual(a,b)
print("Độ dài đầu vào:",len(a),len(b),"đầu ra:",len(ans))
assert int(ans,2)==int(a,2)+int(b,2)

## 13. Class Solution đúng để submit

Tên method là **`addBinary`**, kiểu trả `str`. Không trả list, không thêm `0b`, không `print` bên trong lời giải.

In [ ]:
class Solution:
    def addBinary(self,a:str,b:str)->str:
        i,j=len(a)-1,len(b)-1
        carry=0
        out=[]
        while i>=0 or j>=0 or carry:
            total=carry
            if i>=0:
                total+=int(a[i]); i-=1
            if j>=0:
                total+=int(b[j]); j-=1
            out.append(str(total%2))
            carry=total//2
        return "".join(reversed(out))

assert Solution().addBinary("11","1")=="100"

## 14. Kiểm thử đủ ca nhỏ và nhiều số nhớ

Test độ dài khác nhau, carry ở cuối, chuỗi chỉ có 0/1, các ví dụ LeetCode. Với các số **nhỏ**, có thể dùng `bin(int(a,2)+int(b,2))[2:]` để làm oracle kiểm chứng.

In [ ]:
cases=[("11","1","100"),("1010","1011","10101"),("0","0","0"),
       ("1","0","1"),("1","1","10"),("1111","1","10000"),
       ("100","110010","110110"),("10","10","100")]
for a,b,want in cases:
    got=Solution().addBinary(a,b)
    assert got==want,(a,b,got,want)
    assert got==bin(int(a,2)+int(b,2))[2:]
print("PASS",len(cases),"trường hợp")

## 15. Kiểm tra tất cả cặp nhỏ: đối chiếu độc lập

Các test mẫu có thể chưa đụng mọi nhánh. Ta còn có thể kiểm tra **toàn bộ** 64 cặp số từ 0 tới 7. Để giữ notebook gọn, chỉ in tổng số test đạt.

In [ ]:
for x in range(8):
    for y in range(8):
        a,b=bin(x)[2:],bin(y)[2:]
        assert Solution().addBinary(a,b)==bin(x+y)[2:]
print("PASS 64/64 cặp 0..7")

## 16. Các lỗi dễ gặp

❌ Quên `or carry` trong while: `"1" + "1"` chỉ ra `"0"` thay vì `"10"`.

❌ Cộng trực tiếp ký tự `'1' + '1'`: Python ghép thành chuỗi `'11'`, **không phải số 2**.

❌ Không đảo list `out` trước khi ghép: `"11"+"1"` ra `"001"` thay vì `"100"`.

❌ `total // 10` hoặc `% 10`: bài này **cơ số 2**, không phải hệ thập phân.

✅ `total%2`, `total//2`, và duyệt hết cả carry.

In [ ]:
print("'1' + '1' =",'1'+'1')
print("int('1') + int('1') =",int('1')+int('1'))
print("Đúng:",Solution().addBinary('1','1'))

## 17. Mẹo nhận diện và chuyển giao

Đây là pattern **digit-by-digit arithmetic**: cộng hai dãy chữ số kèm số nhớ. Tư duy từ bài #66 Plus One được dùng lại trong hệ 2, chỉ thay **cơ số** và xử lý **hai** chuỗi đầu vào. Bạn có thể chuyển nó sang hệ 8, 10 hoặc bài cộng hai số rất lớn được lưu bằng chuỗi.

## 18. Năm bài tập tự luyện

### Bài 1 — phần dư và số nhớ — Tự làm

Với total=3 trong hệ 2, trả `(bit, carry)` dưới dạng tuple.

**Gợi ý cấp 1 (tư duy):** Tổng 3 = 1 + 2.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

%2 và //2.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Trả (total%2,total//2).

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_67_1(total:int)->tuple[int,int]:
    # TODO
    pass

# assert exercise_67_1(3)==(1,1)

### Bài 2 — tính bằng tay — Tự làm

`"111" + "1"` ra chuỗi nhị phân nào?

**Gợi ý cấp 1 (tư duy):** 7+1=8.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

8 nhị phân là 1000.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Trả "1000".

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_67_2()->str:
    # TODO
    pass

# assert exercise_67_2()=="1000"

### Bài 3 — đổi bit sang int — Tự làm

Viết hàm trả tổng **hai bit** là ký tự '0'/'1'.

**Gợi ý cấp 1 (tư duy):** Ký tự khác số.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

int(a),int(b).

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

return int(a)+int(b).

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_67_3(a:str,b:str)->int:
    # TODO
    pass

# assert exercise_67_3("1","1")==2

### Bài 4 — tự viết addBinary — Tự làm

Viết lại lời giải không dùng `int(a,2)`, không dùng `bin`.

**Gợi ý cấp 1 (tư duy):** Duyệt từ bên phải; giữ carry.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

while i>=0 or j>=0 or carry.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Lấy total%2, carry=total//2; đảo result cuối.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_67_4(a:str,b:str)->str:
    # TODO
    pass

# assert exercise_67_4("11","1")=="100"

### Bài 5 — biến thể: cộng hai số cơ số 10 — Tự làm

Hai chuỗi thập phân không có số 0 ở đầu (trừ "0"). Trả chuỗi tổng; không ép cả số thành int.

**Gợi ý cấp 1 (tư duy):** Giữ hai index và carry như nhị phân.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Nhưng cơ số phải là 10.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

bit=total%10 và carry=total//10; nối kết quả đảo.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_67_5(a:str,b:str)->str:
    # TODO
    pass

# assert exercise_67_5("999","1")=="1000"

## 19. Đáp án tham khảo — để cuối

In [ ]:
def answer_67_1(total):return total%2,total//2
def answer_67_2():return "1000"
def answer_67_3(a,b):return int(a)+int(b)
def answer_67_4(a,b):
    i,j=len(a)-1,len(b)-1;carry=0;out=[]
    while i>=0 or j>=0 or carry:
        total=carry
        if i>=0:total+=int(a[i]);i-=1
        if j>=0:total+=int(b[j]);j-=1
        out.append(str(total%2));carry=total//2
    return "".join(reversed(out))
def answer_67_5(a,b):
    i,j=len(a)-1,len(b)-1;carry=0;out=[]
    while i>=0 or j>=0 or carry:
        total=carry
        if i>=0:total+=int(a[i]);i-=1
        if j>=0:total+=int(b[j]);j-=1
        out.append(str(total%10));carry=total//10
    return "".join(reversed(out))
assert answer_67_1(3)==(1,1)
assert answer_67_2()=="1000"
assert answer_67_3("1","1")==2
assert answer_67_4("11","1")=="100"
assert answer_67_5("999","1")=="1000"
print("Đáp án PASS 5/5")

## Tóm tắt — mẫu tư duy cần nhớ

**Mẫu:** `i=len(a)-1; j=len(b)-1; carry=0`; while còn a/b/carry: cộng bit cùng carry, append `str(total%2)`, cập nhật `carry=total//2`; trả `"".join(reversed(out))`. **Time/Space:** O(max(n,m)).

## Checklist trước khi submit

- [ ] Tôi giải thích được phép cộng 1+1=10₂.
- [ ] Tôi không nhầm string "1" và số 1.
- [ ] Tôi nhớ xử lý carry sau khi hai chuỗi đã hết.
- [ ] Tôi đảo out trước khi trả kết quả.
- [ ] Tôi đã test "11"+"1" và "1010"+"1011".

**Tự kiểm tra lần cuối:** có thể giải thích lời giải bằng lời trước khi xem code? Có tự tính được ít nhất một bảng trace? Nếu chưa, quay lại phần mô phỏng.

*Notebook học tập được biên soạn theo cẩm nang tiếng Việt đính kèm; tiêu đề/đề/ràng buộc bám sát LeetCode. Những biến thể được đánh dấu là mở rộng, không phải yêu cầu đề chính.*